# **Gaia DR3 Variable Stars:** Feature Engineering

## Imports and Configuration

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from astropy import units as u
from astropy.timeseries import LombScargle
from astropy_healpix import HEALPix
from IPython.display import display

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for path in (root / "src", root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gaia_variable_stars.config import SOURCE_FILE, CLEAN_FILE, PHOTOMETRY_FILE, FEATURE_FILE, CLASSES, MIN_POINTS
from gaia_variable_stars.features import MIN_PERIOD, N_PEAKS, fourier_fit, extract
pd.set_option("display.max_columns", 100)

MAX_FAP = 0.01
MAX_DRIFT = 0.25
CORR_LIMIT = 0.90

FE_FEATURE_FILE = FEATURE_FILE.with_name("gaia_features_fe.csv")
FE_VALIDATION_FILE = FEATURE_FILE.with_name("gaia_period_validation_fe.csv")
FE_ORACLE_FILE = FEATURE_FILE.with_name("gaia_features_fe_oracle.csv")

## Load Data

In [3]:
sources = pd.read_csv(SOURCE_FILE)
photometry = pd.read_csv(CLEAN_FILE, dtype={"transit_id": "string"})

source_cols = [
    "source_id", "class_name", "bp_rp_color", "ecl_lat",
    "ra", "dec", "g_mean_magnitude", "gaia_period",
    "gaia_period_origin"
]
photo_cols = [
    "source_id", "g_transit_time", "g_transit_mag",
    "g_transit_flux", "g_transit_flux_error", "variability_flag_g_reject"
]
missing = (set(source_cols) - set(sources)) | (set(photo_cols) - set(photometry))
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if sources["source_id"].duplicated().any():
    raise ValueError("Duplicate source IDs.")
if not sources["class_name"].isin(CLASSES).all():
    raise ValueError("Unexpected variable-star class.")
if not set(photometry["source_id"]).issubset(set(sources["source_id"])):
    raise ValueError("Photometry contains unknown source IDs.")

for col in ["bp_rp_color", "ecl_lat", "ra", "dec", "gaia_period", "g_mean_magnitude"]:
    sources[col] = pd.to_numeric(sources[col], errors="coerce")

print("Sources:", len(sources))
print("Clean observations:", len(photometry))
display(sources["class_name"].value_counts().reindex(CLASSES, fill_value=0))

Sources: 2000
Clean observations: 85243


class_name
RR Lyrae                500
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    500
Name: count, dtype: int64

## Prepare Photometry

In [4]:
observations = photometry.copy()
for col in ["g_transit_time", "g_transit_mag", "g_transit_flux", "g_transit_flux_error"]:
    observations[col] = pd.to_numeric(observations[col], errors="coerce")

valid = (
    np.isfinite(observations[["g_transit_time", "g_transit_mag", "g_transit_flux"]]).all(axis=1)
    & observations["g_transit_flux"].gt(0)
    & observations["variability_flag_g_reject"].astype("string")
      .str.strip().str.lower().isin(["false", "0"])
)
if not valid.all():
    raise ValueError("CLEAN_FILE contains invalid or rejected observations.")
if observations.duplicated(["source_id", "g_transit_time"]).any():
    raise ValueError("Duplicate source timestamps in CLEAN_FILE.")

flux = observations["g_transit_flux"]
error = observations["g_transit_flux_error"]
observations["magnitude_error"] = (
    2.5 / np.log(10) * error / flux
).where(np.isfinite(error) & error.gt(0))
observations = observations.sort_values(
    ["source_id", "g_transit_time"]
).reset_index(drop=True)

print("Valid observations:", len(observations))
print("Valid magnitude errors:", observations["magnitude_error"].notna().sum())

Valid observations: 85243
Valid magnitude errors: 85243


## Extract Features

In [5]:
records, light_curves = extract(observations.rename(columns={"g_transit_time": "time", "g_transit_mag": "magnitude"}))

## Check Extraction

In [6]:
if len(records) != len(sources) or sum(r["n_epochs"] for r in records) != len(observations):
    raise RuntimeError("Extracted source or epoch counts do not match CLEAN_FILE.")

if any(r["searchable"] and not np.isfinite([r["period_estimate"], r["power"], r["period_fap"]]).all() for r in records):
    raise RuntimeError("Searchable sources have invalid period-search results.")

if not all(np.isfinite(r["amplitude"]) and r["amplitude"] >= 0 for r in records):
    raise RuntimeError("Invalid photometric amplitudes.")

bad_periods = [
    (r["source_id"], r["period_estimate"], r["search_max_period"], r["log_period"])
    for r in records if r["searchable"] and not (
        (r["period_estimate"] >= MIN_PERIOD or np.isclose(r["period_estimate"], MIN_PERIOD, rtol=1e-10, atol=0))
        and (r["period_estimate"] <= r["search_max_period"] or np.isclose(r["period_estimate"], r["search_max_period"], rtol=1e-10, atol=0))
        and np.isclose(np.log10(r["period_estimate"]), r["log_period"], rtol=0, atol=1e-10)
    )
]

if bad_periods:
    raise RuntimeError(f"Invalid period boundaries or log_period for {len(bad_periods)} sources: {bad_periods[:5]}")

print("Extracted Sources:", len(records))
print("Searchable Sources:", sum(r["searchable"] for r in records))

Extracted Sources: 2000
Searchable Sources: 1978


## Feature Tables

In [7]:
stars = sources[["source_id", "class_name", "bp_rp_color", "ecl_lat", "ra", "dec",
                 "g_mean_magnitude", "gaia_period", "gaia_period_origin"]].merge(
    pd.DataFrame(records), on="source_id", how="left", validate="one_to_one"
)
stars["abs_latitude"] = stars["ecl_lat"].abs()
stars["period_reliable"] = stars["period_fap"].le(MAX_FAP)

period_free_cols = [
    "magnitude_mean", "magnitude_std", "magnitude_skew", "magnitude_kurt", "magnitude_mad",
    "amplitude", "median_error", "amplitude_ratio", "bp_rp_color",
    "beyond_1sigma_fraction", "von_neumann_eta", "stetson_j",
    "sf_short", "sf_long", "sf_ratio", "max_visit_slope",
]

period_cols = [
    "log_period", "power", "half_power", "double_power", "half_power_ratio", "double_power_ratio",
    "fourier_amp", "fourier_ratio", "peak_ratio_2_1", "peak_to_median",
    *[f"peak_{kind}_{k}" for k in range(2, N_PEAKS + 1) for kind in ["period", "power"]],
    *[f"best_{band}_{kind}" for band in ["short", "medium", "long"] for kind in ["period", "power"]],
    *[f"{field}_naive" for field in ["phi21", "R31", "phi31"]],
]

sampling_cols = [
    "n_epochs", "time_span", "median_gap", "short_gap_fraction",
    "n_visits", "epochs_per_visit", "median_visit_gap", "visit_gap_cv",
]

predictor_cols = period_free_cols + period_cols
stars[predictor_cols + sampling_cols] = stars[predictor_cols + sampling_cols].replace([np.inf, -np.inf], np.nan)

model_sources = stars["searchable"].fillna(False)
features = stars.loc[model_sources, ["source_id", "class_name", *predictor_cols]].copy()
features.loc[~stars.loc[model_sources, "period_reliable"].to_numpy(), period_cols] = np.nan

excluded = {"gaia_period", "gaia_period_origin", "abs_latitude", "period_eligible",
            "period_status", "half_period_alias", "double_period_alias", "recovered"}
if excluded.intersection(features.columns):
    raise ValueError("Validation-only columns entered the classification predictors.")
if not all(pd.api.types.is_numeric_dtype(features[col]) for col in predictor_cols):
    raise ValueError("Nonnumeric predictors.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite predictors.")
if not features["source_id"].is_unique:
    raise ValueError("Duplicate model rows.")

display(pd.DataFrame({
    "not_searchable": (~model_sources).groupby(stars["class_name"]).sum(),
    "no_reliable_period": (model_sources & ~stars["period_reliable"]).groupby(stars["class_name"]).sum(),
}).reindex(CLASSES))

,not_searchable,no_reliable_period
class_name,,
RR Lyrae,22,172
Cepheid,0,4
Eclipsing Binary,0,165
Long-Period Variable,0,114


## Period Recovery

In [8]:
def period_match(estimate, reference, span):
    estimate, reference, span = np.broadcast_arrays(
        np.asarray(estimate, dtype=float),
        np.asarray(reference, dtype=float),
        np.asarray(span, dtype=float)
    )
    valid = (
        np.isfinite(estimate) & (estimate > 0)
        & np.isfinite(reference) & (reference > 0)
        & np.isfinite(span) & (span > 0)
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        drift = span * np.abs(1 / estimate - 1 / reference)
    return valid & (drift <= MAX_DRIFT)

estimate = stars["period_estimate"].to_numpy(dtype=float)
published = stars["gaia_period"].to_numpy(dtype=float)
span = stars["time_span"].to_numpy(dtype=float)
maximum = stars["search_max_period"].to_numpy(dtype=float)

eligible = (
    stars["searchable"].fillna(False).to_numpy(dtype=bool)
    & np.isfinite(published) & (published >= MIN_PERIOD)
    & (published <= maximum)
)

exact = eligible & period_match(estimate, published, span)
half = eligible & ~exact & period_match(estimate, published / 2, span)
double = eligible & ~exact & ~half & period_match(estimate, published * 2, span)

stars["period_eligible"] = eligible
stars["phase_coherent_recovery"] = exact
stars["half_period_alias"] = half
stars["double_period_alias"] = double
stars["period_status"] = np.select(
    [~eligible, exact, half, double],
    ["ineligible", "exact", "half", "double"], default="mismatch"
)
with np.errstate(divide="ignore", invalid="ignore"):
    stars["primary_relative_error"] = np.where(
        eligible, np.abs(estimate / published - 1), np.nan
    )
    stars["primary_phase_drift_cycles"] = np.where(
        eligible, span * np.abs(1 / estimate - 1 / published), np.nan
    )
stars["relative_period_accurate"] = (
    eligible & (stars["primary_relative_error"].to_numpy() <= 0.10)
)

for k in [1, 3, 5]:
    recovered = np.zeros(len(stars), dtype=bool)
    for j in range(1, k + 1):
        peak = stars[f"peak_period_{j}"].to_numpy(dtype=float)
        recovered |= period_match(peak, published, span)
    stars[f"recovery_at_{k}"] = eligible & recovered

recovery = pd.crosstab(
    stars.loc[eligible, "class_name"], stars.loc[eligible, "period_status"]
).reindex(index=CLASSES, columns=["exact", "half", "double", "mismatch"], fill_value=0)

display(recovery)
display(
    stars.loc[eligible].groupby("class_name")[
        ["recovery_at_1", "recovery_at_3", "recovery_at_5"]
    ].mean().reindex(CLASSES).round(3)
)

period_status,exact,half,double,mismatch
class_name,,,,
RR Lyrae,392,1,0,85
Cepheid,495,0,0,5
Eclipsing Binary,12,366,2,120
Long-Period Variable,269,0,0,132


,recovery_at_1,recovery_at_3,recovery_at_5
class_name,,,
RR Lyrae,0.820,0.921,0.948
Cepheid,0.990,0.996,0.996
Eclipsing Binary,0.024,0.054,0.062
Long-Period Variable,0.671,0.805,0.863


## Sky Position and Observation Quality

In [9]:
healpix = HEALPix(nside=16, order="ring", frame="icrs")
sky_cell = np.full(len(stars), -1, dtype=int)
valid_sky = stars["ra"].between(0, 360) & stars["dec"].between(-90, 90)
sky_cell[valid_sky] = healpix.lonlat_to_healpix(
    stars.loc[valid_sky, "ra"].to_numpy() * u.deg,
    stars.loc[valid_sky, "dec"].to_numpy() * u.deg
)
stars["sky_cell"] = sky_cell

raw = pd.read_csv(
    PHOTOMETRY_FILE,
    usecols=lambda col: col in {
        "source_id", "band", "variability_flag_g_reject", "rejected_by_variability"
    }
)
if "band" in raw:
    raw = raw.loc[raw["band"].astype("string").str.upper().eq("G")]

flag_col = (
    "variability_flag_g_reject" if "variability_flag_g_reject" in raw
    else "rejected_by_variability"
)
flags = raw[flag_col].astype("string").str.strip().str.lower()
raw["rejected"] = np.select(
    [flags.isin(["true", "1"]), flags.isin(["false", "0"])],
    [1.0, 0.0], default=np.nan
)
rejection = raw.groupby("source_id")["rejected"].mean()
stars["gaia_rejection_fraction"] = stars["source_id"].map(rejection)

print("Sky cells:", stars.loc[stars["sky_cell"].ge(0), "sky_cell"].nunique())
print("Sources with rejection data:", stars["gaia_rejection_fraction"].notna().sum())

Sky cells: 169
Sources with rejection data: 2000


## Cohort Flow

In [10]:
flow = pd.DataFrame({
    "selected": sources["class_name"].value_counts(),
    "retrieved_g_band": sources.loc[sources["source_id"].isin(raw["source_id"]), "class_name"].value_counts(),
    "clean_photometry": sources.loc[sources["source_id"].isin(observations["source_id"]), "class_name"].value_counts(),
    "searchable_modeled": features["class_name"].value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)

bad = stars.loc[~model_sources]
reason = np.select(
    [bad["n_epochs"].lt(MIN_POINTS), (bad["time_span"] / 2).le(MIN_PERIOD)],
    ["n_epochs < MIN_POINTS", "baseline/2 <= MIN_PERIOD"],
    default="constant magnitude or too few unique times",
)
flow = flow.join(pd.crosstab(bad["class_name"], reason).reindex(CLASSES, fill_value=0))
flow.to_csv(FEATURE_FILE.with_name("gaia_cohort_flow.csv"))
display(flow)

,selected,retrieved_g_band,clean_photometry,searchable_modeled,n_epochs < MIN_POINTS
class_name,,,,,
RR Lyrae,500,500,500,478,22
Cepheid,500,500,500,500,0
Eclipsing Binary,500,500,500,500,0
Long-Period Variable,500,500,500,500,0


## Period Validation

In [11]:
with np.errstate(divide="ignore", invalid="ignore"):
    stars["n_cycles"] = stars["time_span"] / stars["gaia_period"]
    stars["epochs_per_cycle"] = stars["n_epochs"] / stars["n_cycles"]
    stars["visits_per_cycle"] = stars["n_visits"] / stars["n_cycles"]

validation_cols = [
    "source_id", "class_name", "gaia_period", "gaia_period_origin",
    "period_estimate", "period_fap", "period_reliable",
    "g_mean_magnitude",
    "ecl_lat", "abs_latitude", "sky_cell", "gaia_rejection_fraction",
    "n_cycles", "epochs_per_cycle", "visits_per_cycle",
    *sampling_cols, "search_max_period", "near_search_boundary",
    "period_eligible", "primary_relative_error",
    "primary_phase_drift_cycles", "relative_period_accurate",
    "phase_coherent_recovery", "half_period_alias", "double_period_alias",
    "period_status", "recovery_at_1", "recovery_at_3", "recovery_at_5",
    *[f"peak_{kind}_{k}" for k in range(1, N_PEAKS + 1) for kind in ["period", "power"]]
]
period_validation = stars[validation_cols].replace([np.inf, -np.inf], np.nan).copy()

display(
    period_validation.loc[period_validation["period_eligible"]]
    .groupby("class_name")
    .agg(
        stars=("source_id", "size"),
        median_cycles=("n_cycles", "median"),
        median_epochs=("n_epochs", "median"),
        recovery=("phase_coherent_recovery", "mean"),
        reliable_period=("period_reliable", "mean"),
    ).reindex(CLASSES).round(3)
)

,stars,median_cycles,median_epochs,recovery,reliable_period
class_name,,,,,
RR Lyrae,478,1967.233,37.0,0.820,0.640
Cepheid,500,305.746,39.0,0.990,0.992
Eclipsing Binary,500,2243.189,40.0,0.024,0.670
Long-Period Variable,401,5.183,42.0,0.671,0.793


## Oracle Features

In [12]:
oracle_records = []
for _, star in stars.loc[eligible].iterrows():
    time, mag, errors = light_curves[star["source_id"]]
    period = float(star["gaia_period"])

    search = LombScargle(time - time.min(), mag, normalization="standard")
    power = float(search.power(1 / period))
    half_power = float(search.power(2 / period))
    double_power = float(search.power(1 / (2 * period)))

    naive = fourier_fit(time, mag, period)

    row = {
        "source_id": star["source_id"], "class_name": star["class_name"],
        "gaia_period": period, "log_period": np.log10(period), "power": power,
        "half_power": half_power, "double_power": double_power,
        "half_power_ratio": half_power / power if power > 0 else np.nan,
        "double_power_ratio": double_power / power if power > 0 else np.nan,
        "fourier_amp": naive["amps"].get(1, np.nan) if naive else np.nan,
        "fourier_ratio": naive.get("R21", np.nan) if naive else np.nan,
    }
    for field in ["R21", "phi21", "R31", "phi31"]:
        row[f"{field}_naive"] = naive.get(field, np.nan) if naive else np.nan
    oracle_records.append(row)

oracle = pd.DataFrame(oracle_records).replace([np.inf, -np.inf], np.nan)


print("Oracle sources:", len(oracle))
display(oracle.groupby("class_name").size().reindex(CLASSES, fill_value=0))

Oracle sources: 1879


class_name
RR Lyrae                478
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    401
dtype: int64

## Check Features

In [13]:
missing = features[predictor_cols].isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)
display(pd.DataFrame({"missing": missing, "missing_pct": (100 * missing / len(features)).round(2)}))

corr = features[predictor_cols].corr(method="spearman").abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
display(pairs[pairs.ge(CORR_LIMIT)].sort_values(ascending=False).rename("abs_corr").to_frame())

if features["source_id"].duplicated().any():
    raise ValueError("Duplicate classification sources.")
if not features["source_id"].isin(stars.loc[stars["searchable"], "source_id"]).all():
    raise ValueError("Ineligible sources entered classification.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite classification predictors.")
if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle coverage differs from eligible sources.")

print("Complete predictor rows:", features[predictor_cols].notna().all(axis=1).sum())

,missing,missing_pct
peak_period_3,455,23.00
peak_power_3,455,23.00
R31_naive,455,23.00
phi21_naive,455,23.00
best_long_power,455,23.00
best_long_period,455,23.00
best_medium_power,455,23.00
best_medium_period,455,23.00
best_short_power,455,23.00
best_short_period,455,23.00


,,abs_corr
double_power,double_power_ratio,0.991026
half_power,half_power_ratio,0.985449
magnitude_std,amplitude,0.979082
peak_power_4,peak_power_5,0.975231
magnitude_std,fourier_amp,0.964796
sf_long,fourier_amp,0.964200
magnitude_std,sf_long,0.962214
amplitude,fourier_amp,0.954590
peak_power_3,peak_power_4,0.951432
amplitude,sf_long,0.940658


Complete predictor rows: 1514


## Export Results

In [14]:
FE_FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(FE_FEATURE_FILE, index=False)
period_validation.to_csv(FE_VALIDATION_FILE, index=False)
oracle.to_csv(FE_ORACLE_FILE, index=False)

## Final Checks

In [15]:
outputs = {
    "features": (FE_FEATURE_FILE, features),
    "validation": (FE_VALIDATION_FILE, period_validation),
    "oracle": (FE_ORACLE_FILE, oracle),
}

for name, (path, original) in outputs.items():
    saved = pd.read_csv(path)
    if len(saved) != len(original):
        raise ValueError(f"{name}: saved row count differs.")
    if list(saved.columns) != list(original.columns):
        raise ValueError(f"{name}: saved columns differ.")
    print(f"{name}: {len(saved)} rows, {len(saved.columns)} columns")

if set(period_validation["source_id"]) != set(sources["source_id"]):
    raise ValueError("Period validation source coverage is incomplete.")
if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle source coverage is incomplete.")

features: 1978 rows, 45 columns
validation: 2000 rows, 46 columns
oracle: 1879 rows, 15 columns
